# FM Energy Workshop — METER_01 Forecasting

**Goal:** Predict the next hour's electricity consumption (`electricity_kwh`) for METER_01  
using a Gradient Boosting model trained on lag features and calendar features.

Stages covered:
1. Understand the data
2. Prepare features
3. Train / test split
4. Train GradientBoostingRegressor
5. Evaluate vs. a naïve baseline
6. Plot predictions and save artefacts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, joblib, pathlib
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Paths
DATA_PATH   = pathlib.Path('../data/one_meter.csv')
MODELS_DIR  = pathlib.Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 · Understand the data

Load `one_meter.csv` and get a first look at its shape, types, value ranges, and any missing values.

In [ ]:
raw = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
print(f'Shape: {raw.shape}')          # rows × columns
raw.head()

In [ ]:
print('--- Column types ---')
print(raw.dtypes)
print()
print('--- Missing values ---')
print(raw.isnull().sum())
print()
print('--- Unique meters / buildings ---')
print(raw[['meter_id','building_id']].drop_duplicates())

In [ ]:
print('--- Consumption statistics (kWh) ---')
print(raw['electricity_kwh'].describe().round(2))
print()
print(f"Date range: {raw['timestamp'].min()}  →  {raw['timestamp'].max()}")
print(f"Total hours: {len(raw)}  |  Expected for 60 days: {60*24}")

In [ ]:
# Check for duplicate timestamps or unexpected gaps
raw_sorted = raw.sort_values('timestamp').reset_index(drop=True)
diffs = raw_sorted['timestamp'].diff().dropna()
unexpected = diffs[diffs != pd.Timedelta('1h')]
print(f'Duplicate timestamps: {raw["timestamp"].duplicated().sum()}')
print(f'Gaps / irregular steps: {len(unexpected)}')
if len(unexpected):
    print(unexpected)

In [ ]:
# Quick overview plot
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(raw_sorted['timestamp'], raw_sorted['electricity_kwh'], linewidth=0.7, color='steelblue')
ax.set_title('METER_01 — Hourly Electricity Consumption')
ax.set_ylabel('kWh')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

# ── Observations ────────────────────────────────────────────────────────────
# • 1 440 rows, one per hour, no missing values, no gaps.
# • Single meter (METER_01) in a single office building (Rat_office_Annis).
# • Consumption ranges roughly 17–42 kWh with a clear daily cycle:
#   lower overnight, higher during working hours.
# • A visible weekly pattern: lower on weekends than weekdays.
# • No obvious spikes or zero readings — the data is clean and ready to use.
# ────────────────────────────────────────────────────────────────────────────

---
## 2 · Prepare the data — build features

For each hour **t** we create:

| Feature | Meaning |
|---|---|
| `lag_1h` | Reading at t−1 |
| `lag_2h` | Reading at t−2 |
| `lag_3h` | Reading at t−3 |
| `lag_24h` | Reading at t−24 (same hour yesterday) |
| `lag_168h` | Reading at t−168 (same hour last week) |
| `hour` | Hour of day (0–23) |
| `dayofweek` | Day of week (0=Monday … 6=Sunday) |

All lags look **backwards only** — no future data leaks in.

The first 168 rows cannot have a full 168-hour history, so we drop them.

In [ ]:
df = raw_sorted[['timestamp', 'electricity_kwh']].copy()

# Lag features — shift(n) moves values DOWN by n rows, so row t gets the value from row t-n
for lag in [1, 2, 3, 24, 168]:
    df[f'lag_{lag}h'] = df['electricity_kwh'].shift(lag)

# Calendar features
df['hour']      = df['timestamp'].dt.hour
df['dayofweek'] = df['timestamp'].dt.dayofweek   # 0=Monday

# Drop the first 168 rows that cannot have a complete lag_168h
df = df.dropna().reset_index(drop=True)

print(f'Rows after dropping incomplete history: {len(df)}')
print(f'Date range: {df["timestamp"].min()}  →  {df["timestamp"].max()}')
df.head(3)

---
## 3 · Train / test split

Hold back the **last two weeks** (2016-02-16 onwards) as the test set.  
The model trains on every row before that date — it never sees test-period readings.

In [ ]:
SPLIT_DATE = '2016-02-16'

FEATURES = ['lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_168h', 'hour', 'dayofweek']
TARGET   = 'electricity_kwh'

train = df[df['timestamp'] <  SPLIT_DATE]
test  = df[df['timestamp'] >= SPLIT_DATE]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train rows: {len(train)}  ({train["timestamp"].min().date()} → {train["timestamp"].max().date()})')
print(f'Test rows:  {len(test)}   ({test["timestamp"].min().date()} → {test["timestamp"].max().date()})')

---
## 4 · Train GradientBoostingRegressor

Gradient boosting builds an ensemble of shallow decision trees sequentially,  
each one correcting the errors of the previous. It handles the non-linear  
daily/weekly patterns in energy data well with minimal configuration.

In [ ]:
model = GradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)
print('Training complete.')

# Feature importances — which inputs matter most?
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print()
print('Feature importances:')
print(importances.round(3).to_string())

---
## 5 · Evaluate the model

We compare the model against a **naïve baseline**: predict the next hour by simply  
repeating the previous hour's reading (`lag_1h`). This is the simplest possible  
forecast — if our model can't beat it, it's not useful.

In [ ]:
y_pred     = model.predict(X_test)
y_baseline = X_test['lag_1h'].values   # naïve: predict = last hour's reading

def metrics(actual, predicted, label):
    mae  = mean_absolute_error(actual, predicted)
    rmse = root_mean_squared_error(actual, predicted)
    print(f'{label:<30}  MAE = {mae:.3f} kWh   RMSE = {rmse:.3f} kWh')
    return mae, rmse

print(f'{"Model":<30}  {"MAE":>12}   {"RMSE":>12}')
print('-' * 58)
m_mae,  m_rmse  = metrics(y_test, y_pred,     'GradientBoostingRegressor')
b_mae,  b_rmse  = metrics(y_test, y_baseline, 'Naïve baseline (lag_1h)')
print()
print(f'MAE  improvement over baseline: {(1 - m_mae/b_mae)*100:.1f}%')
print(f'RMSE improvement over baseline: {(1 - m_rmse/b_rmse)*100:.1f}%')

---
## 6 · Plot predictions and save artefacts

In [ ]:
results = test[['timestamp']].copy().reset_index(drop=True)
results['actual']    = y_test.values
results['predicted'] = y_pred
results['error']     = results['predicted'] - results['actual']

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Top panel: actual vs predicted
axes[0].plot(results['timestamp'], results['actual'],    label='Actual',    linewidth=1.0, color='steelblue')
axes[0].plot(results['timestamp'], results['predicted'], label='Predicted', linewidth=1.0, color='darkorange', linestyle='--')
axes[0].set_title('METER_01 — Actual vs Predicted (test period: last two weeks)')
axes[0].set_ylabel('kWh')
axes[0].legend()

# Bottom panel: prediction error
axes[1].fill_between(results['timestamp'], results['error'], 0,
                     where=results['error'] >= 0, color='darkorange', alpha=0.4, label='Over-predicted')
axes[1].fill_between(results['timestamp'], results['error'], 0,
                     where=results['error'] <  0, color='steelblue',  alpha=0.4, label='Under-predicted')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Prediction error (predicted − actual)')
axes[1].set_ylabel('kWh')
axes[1].set_xlabel('Date')
axes[1].legend()

plt.tight_layout()
plt.show()

# ── Interpretation ───────────────────────────────────────────────────────────
# The model tracks the daily rhythm closely — the orange dashed line follows
# the blue line through peaks (daytime) and troughs (overnight).
#
# The error panel shows errors are roughly centred around zero with no
# systematic drift, meaning the model is not consistently biased high or low.
#
# Larger errors tend to cluster around transitions (e.g. Monday morning
# switch-on) where consumption changes quickly — this is typical for
# lag-based models and can be improved with occupancy or calendar features.
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Save the fitted model
model_path = MODELS_DIR / 'gbr_model.joblib'
joblib.dump(model, model_path)
print(f'Model saved → {model_path}')

# Save the feature order so the dashboard assembles inputs in the right sequence
feature_path = MODELS_DIR / 'feature_order.json'
with open(feature_path, 'w') as f:
    json.dump(FEATURES, f, indent=2)
print(f'Feature order saved → {feature_path}')
print(f'Features: {FEATURES}')